# NeuroProfile on Colab


## 0 · GPU check

In [ ]:
import torch
try:
    from tribev2 import TribeModel
    import neuralset
    print(">>> IMPORT OK — tribev2 loads on torch", torch.__version__)
except Exception as e:
    import traceback; traceback.print_exc()
    print("\n>>> IMPORT FAILED:", type(e).__name__, "-", e)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1 · Installs — run this, then **restart the runtime once**

All pip installs up front. After this cell: **Runtime ▸ Restart session**, then continue from Section 2. Restarting makes the pinned torch/numpy the versions that actually load (Colab preloads its own torch).

In [ ]:
# numpy + torch must match tribev2's pins (numpy==2.2.6, torch>=2.5.1,<2.7).
# Installing the exact triple matches your handoff; if Colab's stock torch is already
# in [2.5.1, 2.7) you can skip the torch line and let tribev2 accept it.
!pip install -q numpy==2.2.6
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

# TRIBE v2 (not on PyPI). Its pins match ours, so it won't swap the torch triple.
!pip install -q "git+https://github.com/facebookresearch/tribev2.git"

# CPU-side pipeline deps (your repo's requirements, listed for a clean Colab env)
!pip install -q nibabel qdrant-client fastapi python-multipart uvicorn yt-dlp pytest scipy

# whisperX installed IN THIS env so TRIBE calls it directly (patched below) instead of
# uvx re-downloading ~3.5 GB every call. If this bumps torch, re-run the torch line above.
!pip install -q whisperx

print("installs done — now Runtime > Restart session, then run Section 2")

### ⚠️ Restart the runtime now (Runtime ▸ Restart session), then run Section 2 onward.

## 2 · Drive, repo, auth, weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
NP = '/content/drive/MyDrive/neuroprofile'   # durable root (survives disconnects)
for sub in ('qdrant_data', 'data/timelines', 'clips'):
    os.makedirs(f'{NP}/{sub}', exist_ok=True)
print('durable root:', NP)

In [ ]:
# Get the repo onto FAST local scratch (/content), not Drive (Drive FUSE is slow for code).
# Make sure ica/ (frozen artifacts) and tests/reducer_reference.npz come along —
# reducer.py does np.load("ica/...") at import time and will crash without them.

# --- Option A: clone from GitHub (fill in your remote) ---
!git pull
!git clone https://github.com/Mammbo/NeuroProfile.git /content/neuroprofile

# --- Option B: repo already in Drive — copy to scratch ---
# !cp -r /content/drive/MyDrive/neuroprofile/repo /content/neuroprofile

%cd /content/neuroprofile
!ls backend ica batch_encoding/

In [ ]:
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("Paste HF read token: ").strip()

# verify it works:
from huggingface_hub import HfApi
print("auth OK as", HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"])

In [ ]:
# Colab's network is fast — this is the step that swung 47min->5.5hr on the 4060.
!hf download meta-llama/Llama-3.2-3B --include "*.safetensors" "config.json" "tokenizer*"
# older huggingface_hub: !huggingface-cli download meta-llama/Llama-3.2-3B --include "*.safetensors" "config.json" "tokenizer*"

In [ ]:
import os
os.environ["NLTK_ALLOW_PROXIED_URLOPEN"] = "1"
import nltk
for r in ["punkt_tab", "punkt"]:
    nltk.download(r)

# Patch TRIBE so get_events_dataframe calls whisperx directly instead of via `uvx`
# (uvx spins an isolated env and re-downloads ~3.5 GB every call -> looks frozen at 0%).
import tribev2, pathlib
et  = pathlib.Path(tribev2.__file__).parent / "eventstransforms.py"
src = et.read_text()
new = src.replace('["uvx", "whisperx"', '["whisperx"')
et.write_text(new)
print("uvx->whisperx patch:", "applied" if new != src else "NO CHANGE — check the pattern in eventstransforms.py")

# NOTE: no ctranslate2 execstack ELF-patch needed on Colab (Ubuntu kernel allows exec-stack).
# TRIBE's default whisperx is large-v3 fp16 on cuda. On a 16 GB T4 it may fit alongside TRIBE.
# If predict()+whisperx OOM even here, apply your 4060 edit (model="small", device="cpu",
# compute_type="int8") to eventstransforms.py.

## 3 · Feed clips (upload `.mp4`s to Drive, then encode)

Upload your pre-downloaded clips to `MyDrive/neuroprofile/clips/`. The **file route** needs no yt-dlp, no cookies, no download — `resolve_source` sniffs magic bytes and goes straight to `chunk_video`.

In [ ]:
!pip install -q ffmpeg-python

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/neuroprofile/clips', exist_ok=True)
!ls -la /content/drive/MyDrive/neuroprofile/clips

In [ ]:
%cd /content/neuroprofile
!python batch_encoding/test_encode.py /content/drive/MyDrive/neuroprofile/clips/test1.mp4

In [ ]:
# Build a corpus of FILE PATHS (not URLs) and grind it, persisting to Drive.
import glob, pathlib
clips = sorted(glob.glob('/content/drive/MyDrive/neuroprofile/clips/*.mp4'))
pathlib.Path('/content/corpus.txt').write_text("\n".join(clips))
print(len(clips), "clips -> /content/corpus.txt")

In [ ]:
#TEST run with 2 clips 
%cd /content/neuroprofile
!python batch_encoding/batch_encode.py --corpus /content/corpus.txt --limit 2 \
    --qdrant-path   /content/drive/MyDrive/neuroprofile/qdrant_data \
    --timelines-dir /content/drive/MyDrive/neuroprofile/data/timelines


In [ ]:
from qdrant_client import QdrantClient
QP = "/content/drive/MyDrive/neuroprofile/qdrant_data"
client = QdrantClient(path=QP)

print("collections:", [c.name for c in client.get_collections().collections])
print("count:", client.count("videos_v1").count)

pts, _ = client.scroll("videos_v1", limit=20, with_payload=True, with_vectors=False)
for p in pts:
    pl = p.payload
    print("\n—", pl["video_id"], "|", pl["title"], "| dur", pl.get("duration"), "s")
    print("   profile:", [round(x, 3) for x in pl["system_profile"]])
    print("   systems:", pl["system_names"])
    print("   moments:", len(pl.get("moments", [])), "| timeline:", pl["timeline_path"])

client.close()   # important — embedded Qdrant is single-process; close before anything else opens it

In [ ]:
!python batch_encoding/batch_encode.py --corpus /content/corpus.txt \
    --qdrant-path   /content/drive/MyDrive/neuroprofile/qdrant_data \
    --timelines-dir /content/drive/MyDrive/neuroprofile/data/timelines

# Persisted to Drive => a Colab disconnect mid-corpus is fine: re-run this cell and
# already-encoded ids are skipped (db.get_video != None -> status "skip"). 

live inference

In [ ]:
# 0.
%cd /content/neuroprofile
!git pull

# 1. kill the old server + tunnel if they're still running
!pkill -f analyze_server.py 2>/dev/null; pkill -f cloudflared 2>/dev/null
import time; time.sleep(2)

# 2. cloudflared (safe to re-run; no-op if already installed)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

# 3. start the backend 
import os, subprocess
subprocess.Popen(
    ["python","batch_encoding/analyze_server.py",
     "--qdrant-path","/content/drive/MyDrive/neuroprofile/qdrant_data",
     "--timelines-dir","/content/drive/MyDrive/neuroprofile/data/timelines",
     "--videos-dir","/content/drive/MyDrive/neuroprofile/videos",
     "--host","127.0.0.1","--port","8000"],
    cwd="/content/neuroprofile", env=dict(os.environ),
    stdout=open("/content/analyze.log","a"), stderr=subprocess.STDOUT)
time.sleep(8); print(open("/content/analyze.log").read()[-800:])

# 4. tunnel 
!cloudflared tunnel --url http://localhost:8000